In [4]:
"""
Simplified spike-data extraction script.

"""

import os
import glob
import json
import pickle
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# paths (edit these if your data locations differ)
# ---------------------------------------------------------------------------
bhv_data_root = "/gpfs/marilyn/pi/nandy/Jamie/marmoset_behavioral_data/"

neural_data_folder = "/gpfs/radev/pi/nandy/jadi_gibbs_data/Marmoset_neural_recording/"

data_saved_folder = "/gpfs/marilyn/pi/nandy/Jamie/analysis_result_folder/"

fs_spikes = 20000  # neural recording sampling rate (Hz)


In [18]:

# ---------------------------------------------------------------------------
# session metadata lookup: neural_record_condition -> session info
#
# THIS IS THE ONLY THING YOU SHOULD NEED TO EDIT.
# Keyed by `neural_record_condition` (unique per animal per recording),
# NOT by date -- the same date_tgt can show up twice when two animals are
# recorded in the same session, so it can't be used as a dict key.
# `date_tgt`/`videodate_tgt` are stored as fields instead, for building the
# behavioral file paths.
# `trig_channelname` is the NI trigger channel used to align neural time to
# session start for that recording -- update it freely per session.
# ---------------------------------------------------------------------------

def _build_session_metadata():
    meta = {}

    # ---- Cubone ----
    # the list for neural recording data (from the logger)
    neural_record_conditions = [
        '20260709_Cubone_MC_withDitto',
    ]
    # the list for behavioral data (from the task code)
    dates_list = [
        '20260709_MC',
    ]
    # the list for the video and DLC tracking (from the camera)
    videodates_list = [
        '20260709_MC',
    ]
    # the session start time from the video (when is the beep sound and the first led flash)
    session_start_times = [
        10,
    ]
    trig_channelnames = [
        'Dev1/ai9',
                        ] # transcevier 1  is 'Dev1/ai0', and transceiver is 'Dev1/ai9'

    kilosortvers = [4] * len(dates_list)
    animal1_fixedorders = ['cubone'] * len(dates_list)
    animal2_fixedorders = ['ditto'] * len(dates_list)
    animal1_filenames = ["Cubone"] * len(dates_list)
    animal2_filenames = ["Ditto"] * len(dates_list)
    recorded_animals = ['cubone'] * len(dates_list) # make sure it's cubone


    for i, d in enumerate(neural_record_conditions):
        meta[d] = dict(
            neural_record_condition=neural_record_conditions[i],
            date_tgt=dates_list[i],
            videodate_tgt=videodates_list[i],
            kilosortver=kilosortvers[i],
            session_start_time=session_start_times[i],
            animal1_fixedorder=animal1_fixedorders[i],
            animal2_fixedorder=animal2_fixedorders[i],
            animal1_filename=animal1_filenames[i],
            animal2_filename=animal2_filenames[i],
            recorded_animal=recorded_animals[i],
            trig_channelname=trig_channelnames[i],
        )


    # ---- Ditto ----
    neural_record_conditions = [
        '20260709_Ditto_MC_withCubone',
    ]
    dates_list = [
        '20260709_MC',
    ]
    videodates_list = [
        '20260709_MC',
    ]
    session_start_times = [
        10,
    ]
    trig_channelnames = [
        'Dev1/ai0',
                        ] # transcevier 1  is 'Dev1/ai0', and transceiver is 'Dev1/ai9'

    kilosortvers = [4] * len(dates_list)
    animal1_fixedorders = ['cubone'] * len(dates_list)
    animal2_fixedorders = ['ditto'] * len(dates_list)
    animal1_filenames = ["Cubone"] * len(dates_list)
    animal2_filenames = ["Ditto"] * len(dates_list)
    recorded_animals = ['ditto'] * len(dates_list) # make sure it's ditto


    for i, d in enumerate(neural_record_conditions):
        meta[d] = dict(
            neural_record_condition=neural_record_conditions[i],
            date_tgt=dates_list[i],
            videodate_tgt=videodates_list[i],
            kilosortver=kilosortvers[i],
            session_start_time=session_start_times[i],
            animal1_fixedorder=animal1_fixedorders[i],
            animal2_fixedorder=animal2_fixedorders[i],
            animal1_filename=animal1_filenames[i],
            animal2_filename=animal2_filenames[i],
            recorded_animal=recorded_animals[i],
            trig_channelname=trig_channelnames[i],
        )

    return meta

SESSION_METADATA = _build_session_metadata()



# ---------------------------------------------------------------------------
# core extraction
# ---------------------------------------------------------------------------

def _load_ni_data(date_tgt, animal1_filename, animal2_filename):
    """Locate and load the ni_data json used to align neural time to session
    start. Tries both animal-name orderings, matching the original notebook."""
    bhv_data_path = f"{bhv_data_root}{date_tgt}_{animal1_filename}_{animal2_filename}/"
    for a, b in [(animal2_filename, animal1_filename), (animal1_filename, animal2_filename)]:
        ni_data_json = glob.glob(bhv_data_path + f"{date_tgt}_{a}_{b}_ni_data_*.json")
        if ni_data_json:
            with open(ni_data_json[0]) as f:
                for line in f:
                    return json.loads(line)
    return None


def _get_trigger_elapsed_time(ni_data, trig_channelname):
    """Look up the elapsed_time for the given trigger channel name.

    Expected structure (kilosort4 sessions):
        ni_data['trigger_ts'] is a list of entries, each with a
        'channel_name' and a 'timepoints' list; we take the first
        timepoint's elapsed_time for the matching channel.
    """
    return next(
        entry['timepoints'][-1]['elapsed_time']
        for entry in ni_data['trigger_ts']
        if entry['channel_name'] == f"{trig_channelname}"
    )


def extract_spike_data(target_conditions=None, save=False):
    """
    Extract curated ('good'/'mua') spike times and cluster ids for each
    recording in `target_conditions`.

    Parameters
    ----------
    target_conditions : list of str, optional
        `neural_record_condition` values to process, matching the keys in
        SESSION_METADATA (e.g. "20260709_Cubone_MC_withDitto"). If None
        (default), processes every recording currently defined in
        SESSION_METADATA -- so you only need to edit the metadata dict
        above, not this call.
    save : bool
        If True, pickle the result dict to `data_saved_folder`.

    Returns
    -------
    dict : neural_record_condition -> spike data (see module docstring)
    """
    if target_conditions is None:
        target_conditions = list(SESSION_METADATA.keys())

    spike_data_all_conditions = {}

    for cond_tgt in target_conditions:
        if cond_tgt not in SESSION_METADATA:
            print(f"[skip] '{cond_tgt}' not found in SESSION_METADATA")
            continue

        info = SESSION_METADATA[cond_tgt]
        neural_record_condition = info['neural_record_condition']
        date_tgt = info['date_tgt']
        kilosortver = info['kilosortver']
        trig_channelname = info['trig_channelname']

        print(f"loading spike data for {date_tgt} ({neural_record_condition})")

        # ---- load spike sorting results ----
        ks_subdir = 'Kilosort' if kilosortver == 2 else 'kilosort4'
        base = f"{neural_data_folder}{neural_record_condition}/{ks_subdir}/"

        spike_time_data = np.load(base + 'spike_times.npy')
        spike_clusters_data = np.load(base + 'spike_clusters.npy')
        clusters_info_data = pd.read_csv(base + 'cluster_info.tsv', sep="\t")

        # ---- keep only manually curated units ----
        id_col = 'cluster_id' if 'cluster_id' in clusters_info_data.columns else 'id'
        good_clusters = clusters_info_data[
            (clusters_info_data.group == 'good') | (clusters_info_data.group == 'mua')
        ][id_col].values
        clusters_info_data = clusters_info_data[~pd.isnull(clusters_info_data.group)]

        keep = np.isin(spike_clusters_data, good_clusters)
        spike_time_data = spike_time_data[keep]
        spike_clusters_data = spike_clusters_data[keep]

        # ---- align to session start (best-effort) ----
        aligned = False
        try:
            ni_data = _load_ni_data(date_tgt, info['animal1_filename'], info['animal2_filename'])
            session_start_time_niboard_offset = ni_data['session_t0_offset']
            neural_start_time_niboard_offset = _get_trigger_elapsed_time(ni_data, trig_channelname)
            neural_start_time_session_start_offset = (
                neural_start_time_niboard_offset - session_start_time_niboard_offset
            )
            spike_time_sec = spike_time_data.ravel() / fs_spikes + neural_start_time_session_start_offset
            aligned = True
        except Exception as e:
            print(f"  [warn] could not align to session start ({e}); returning unaligned times")
            spike_time_sec = spike_time_data.ravel() / fs_spikes

        spike_data_all_conditions[cond_tgt] = dict(
            date_tgt=date_tgt,
            recorded_animal=info['recorded_animal'],
            spike_times_sec=spike_time_sec,
            spike_clusters=spike_clusters_data.ravel(),
            good_clusters=good_clusters,
            clusters_info=clusters_info_data,
            aligned=aligned,
        )

    if save:
        out_dir = os.path.join(data_saved_folder, 'spike_data_only')
        os.makedirs(out_dir, exist_ok=True)
        out_path = os.path.join(out_dir, 'spike_data_all_conditions.pkl')
        with open(out_path, 'wb') as f:
            pickle.dump(spike_data_all_conditions, f)
        print(f"saved to {out_path}")

    return spike_data_all_conditions


In [19]:
if __name__ == "__main__":
    # no need to list anything here -- extract_spike_data() defaults to every
    # recording currently defined in SESSION_METADATA. Pass an explicit list
    # of neural_record_condition strings only if you want to run a subset.
    result = extract_spike_data(save=False)
    for cond, v in result.items():
        print(cond, "(", v['date_tgt'], v['recorded_animal'], ") -> n_spikes:", 
              len(v['spike_times_sec']), "n_units:", len(v['good_clusters']), "aligned:", v['aligned'])


loading spike data for 20260709_MC (20260709_Cubone_MC_withDitto)
loading spike data for 20260709_MC (20260709_Ditto_MC_withCubone)
20260709_Cubone_MC_withDitto ( 20260709_MC cubone ) -> n_spikes: 112558 n_units: 51 aligned: True
20260709_Ditto_MC_withCubone ( 20260709_MC ditto ) -> n_spikes: 277191 n_units: 28 aligned: True
